<a href="https://colab.research.google.com/github/mdondzik/air-quality-interpolation/blob/main/01_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Collection – Warsaw PM2.5

This notebook focuses on collecting real-world air quality measurements for Warsaw. The data is sourced from the OpenAQ platform, which aggregates environmental sensor readings from multiple monitoring stations.  

The goal is to obtain a **clean, reproducible dataset** of avarage PM2.5 concentrations across the city in winter months (december 2024-april 2025), suitable for spatial analysis and interpolation. Steps include:

- Retrieving measurements for all Warsaw stations within a selected time period  
- Filtering invalid or extreme values  
- Aggregating multiple measurements per location to produce a single representative value  

This dataset forms the foundation for the subsequent analysis, visualization, and modeling in this project.

Finding all Warsaw (and neighbouring) stations:

In [2]:
!pip install openaq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 2.2 MB/s eta 0:00:00


In [3]:
import pandas as pd
from openaq import OpenAQ

api = OpenAQ(api_key="4aa2b8f0762a835f4d664afb82b1da54afaee6093393628252ff31f87309e51a")

locations = api.locations.list(
    bbox=(20.8, 52.0, 21.3, 52.4),  # Warsaw bounds
    parameters_id=2,
    limit=1000
)

print(f"{len(locations.results)} locations in Warsaw")

24 locations in Warsaw


Choosing sensors measuring PM2.5:

In [4]:
sensor_data = []

for loc in locations.results:
    for sensor in loc.sensors:
        if sensor.parameter.id == 2:  # PM2.5
            sensor_data.append({
                "sensor_id": sensor.id,
                "lat": loc.coordinates.latitude,
                "lon": loc.coordinates.longitude
            })

print(f"{len(sensor_data)} PM2.5 sensors")

25 PM2.5 sensors


Fetching monthly avarage PM2.5 for each sensor:

In [5]:
all_data = []

for s in sensor_data:
    measurements = api.measurements.list(
        sensors_id=s["sensor_id"],
        data="days",
        rollup="monthly",
        limit=1000
    )
    for m in measurements.results:
        all_data.append({
            "lat": s["lat"],
            "lon": s["lon"],
            "measurements": m
        })
print(f"{len(all_data)} monthly avarage measurements")

899 monthly avarage measurements


Filtering data by date:

In [10]:
target_start = "2024-12-01"
target_end = "2025-04-30"

filtered = [
    m for m in all_data
    if target_start <= str(m["measurements"].period.datetime_from.local)[:10] <= target_end
]

df = pd.DataFrame([{
    "lat": m["lat"],
    "lon": m["lon"],
    "value": m["measurements"].value
} for m in filtered])

print(f"{len(df)} measurements between december 2024 and april 2025")

66 measurements between december 2024 and april 2025


Filtering out invalid or extreme data:

In [7]:
df = df[(df["value"] > 0) & (df["value"] < 300)]

print(f"{len(df)} valid measurements")

66 valid measurements


Aggregating data:

In [9]:
df = df.groupby(["lat", "lon"]).mean().reset_index()
print(df.head())
print(f"{len(df)} locations with final average PM2.5")

         lat        lon  value
0  52.082277  21.124598  21.54
1  52.098304  21.040840  20.20
2  52.115725  21.237297  33.56
3  52.143305  21.066756  35.52
4  52.160772  21.033819  18.85
14 locations with final average PM2.5


Saving final database:

In [11]:
df.to_csv("pm25_database.csv", index=False)